# Reinforcement Learning for HR Onboarding: Teaching an LLM to Automate Enterprise Workflows

In this tutorial, we'll train an LLM to complete **HR onboarding and offboarding tasks** using **reinforcement learning (RL)**. The agent learns to generate JSON tool calls to complete multi-step workflows like:

- Creating employee records and initiating onboarding
- Assigning laptops, provisioning IT accounts, setting up access roles
- Sending welcome emails, scheduling orientation meetings
- Processing offboarding with asset reclaim and access revocation

By the end, you'll understand how to:
- Connect LLMs to enterprise environments using **OpenEnv**
- Design reward functions based on **rubric criteria**
- Train models with **GRPO** (Group Relative Policy Optimization)
- Evaluate improvement across simple, medium, complex, and edge-case tasks

**Requirements:** This notebook runs on a free Tesla T4 Google Colab instance (or any GPU with 16GB+ VRAM).

**Baseline (GPT-4o-mini):** 50.6% pass rate, 0.791 mean score. Let's see how RL training improves an open-source model!

## What is the HR Onboarding Environment?

This is an **OpenEnv-compatible RL environment** that simulates the HR department of a fictional company called **AcmeCorp**. It has:

- **200 employees** across 8 departments with a full org hierarchy (L1-L6 levels)
- **25 tools** the agent can call (HR, IT, access control, communication, policy)
- **77 tasks** across 4 difficulties (simple, medium, complex, edge case)
- **Rubric-based rewards** — each task has verifiable criteria (did you call the right tool? with the right params? in the right order?)

### Our Goal

The agent receives a task instruction (e.g., "Onboard Priya Sharma to Engineering as L2 Software Engineer") and must generate a **sequence of JSON tool calls** to complete it. Each tool call is one step. The agent has up to 15 steps per episode.

Unlike the 2048 tutorial where the model writes code, here the model **directly generates tool calls** — closer to how real enterprise agents work.

## Installation

We need:
1. **[Unsloth](https://github.com/unslothai/unsloth)** — Memory-efficient LLM training (~70% less VRAM)
2. **[TRL](https://github.com/huggingface/trl)** — GRPO trainer for RL
3. **Our HR environment** — Cloned from GitHub

In [7]:
%%capture
import os, importlib.util

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    !pip install \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !pip install unsloth trackio

!pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

Next, clone the HR environment and install it:

In [13]:
!pip install openenv-core datasets pydantic python-dotenv regex safetensors transformers unsloth accelerate trackio

/opt/conda/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=1699) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


  Using cached trackio-0.18.0-py3-none-any.whl.metadata (13 kB)
  Using cached gradio-6.9.0-py3-none-any.whl.metadata (16 kB)
  Using cached orjson-3.11.7-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (41 kB)
  Using cached plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (2.0 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached tomlkit-0.13.3-py3-none-any.whl.metadata (2.8 kB)
  Using cached itsdangerous-2.2.0-py3-none-

## Loading the Model

We load the model with memory optimizations to fit on a T4 GPU:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `max_seq_length` | 2048 | Longer context for multi-step tool calling |
| `load_in_4bit` | True | 4-bit quantization to reduce memory |
| `lora_rank` | 8 | LoRA adapter rank (balance of quality vs memory) |

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Longer context for multi-turn tool calling
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    load_in_4bit=True,
    max_seq_length=max_seq_length,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.13/site-packages/triton/runtime/autotuner.py:101: DeprecationWarning: warmup, rep, and use_cuda_graph parameters are deprecated. See https://github.com/triton-lang/triton/pull/4496 for details.
  warnings.warn(("warmup, rep, and use_cuda_graph parameters are deprecated. See "


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/opt/conda/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=1699) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### Applying LoRA for Efficient Training

[LoRA (Low-Rank Adaptation)](https://hf.co/papers/2106.09685) adds small trainable adapters (~1-5% of parameters) instead of updating all weights. We target the attention and feedforward layers:

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## Setting Up the HR Environment

Our environment runs **locally** (no remote server needed). It manages 500+ entities and 25 tools. Let's set it up and see what a task looks like:

In [3]:
import json
import re

from server.hr_onboarding_environment import HROnboardingEnvironment
from models import HROnboardingAction, HROnboardingObservation
from server.tools import TOOL_DEFINITIONS
from server.rubrics import RubricEvaluator

# Create the environment
env = HROnboardingEnvironment(seed=42, max_steps=15)

print(f"Total tasks: {len(env._tasks)}")
print(f"Total tools: {len(TOOL_DEFINITIONS)}")

# Show a sample task
obs = env.reset()
print(f"\nSample task: {obs.task_id}")
print(f"Difficulty: {obs.metadata.get('difficulty')}")
print(f"Category: {obs.metadata.get('category')}")
print(f"Instruction: {obs.instruction}")
print(f"Available tools: {len(obs.available_tools)}")

Total tasks: 77
Total tools: 25

Sample task: task_0001
Difficulty: simple
Category: lookup
Instruction: Look up the employee record for Jennifer Davis (ID: emp_0016).
Available tools: 25


Let's try calling a tool manually to see how the environment works:

In [4]:
# Call a tool
action = HROnboardingAction(
    tool_name="hr_read_employee",
    arguments={"emp_id": "emp_0001"}
)
obs = env.step(action)
print("Tool result:")
print(json.dumps(obs.tool_result, indent=2)[:500])

Tool result:
{
  "success": true,
  "employee": {
    "emp_id": "emp_0001",
    "name": "Rajesh Kumar",
    "email": "rajesh.kumar@acmecorp.com",
    "department": "Engineering",
    "level": "L6",
    "role": "VP of Engineering",
    "manager_id": null,
    "status": "active",
    "date_of_joining": "2018-03-15",
    "date_of_leaving": null,
    "is_contractor": false,
    "phone": "+1-415-332-7891",
    "location": "San Francisco"
  }
}


## Prompt Design

The prompt tells the model what to generate. Unlike the 2048 tutorial (which generates Python code), here the model generates **JSON tool calls** directly:

```json
{"tool": "hr_create_employee", "params": {"name": "Priya Sharma", "department": "Engineering", "level": "L2", "role": "Software Engineer"}}
```

The model gets the task instruction + tool definitions, and must output a sequence of tool calls.

In [5]:
# Build concise tool descriptions for the prompt
tool_desc = json.dumps(TOOL_DEFINITIONS, indent=2)

SYSTEM_PROMPT = (
    "You are an HR automation agent for AcmeCorp. You complete employee "
    "onboarding and offboarding tasks by calling tools.\n\n"
    "For each step, respond with ONLY a JSON tool call:\n"
    '{"tool": "<tool_name>", "params": {<parameters>}}\n\n'
    'When the task is complete, respond with:\n'
    '{"tool": "__done__", "params": {}}\n\n'
    "Rules:\n"
    "- Respond with ONLY the JSON object, no other text\n"
    "- Use exact tool names and parameter names\n"
    "- Create employee records before initiating onboarding\n"
    "- Check asset availability before assigning\n"
    "- Complete all required steps mentioned in the instruction\n\n"
    f"Available tools:\n{tool_desc}"
)

print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")
print(f"System prompt preview: {SYSTEM_PROMPT[:300]}...")

System prompt length: 14997 chars
System prompt preview: You are an HR automation agent for AcmeCorp. You complete employee onboarding and offboarding tasks by calling tools.

For each step, respond with ONLY a JSON tool call:
{"tool": "<tool_name>", "params": {<parameters>}}

When the task is complete, respond with:
{"tool": "__done__", "params": {}}

Ru...


## Building the Training Dataset

Each training example is a task from our environment. We create prompts with the system instructions + task instruction. The model learns to generate good tool call sequences via RL rewards.

In [6]:
from datasets import Dataset

# Build prompts from all tasks
train_prompts = []
train_env = HROnboardingEnvironment(seed=42, max_steps=15)

for i in range(len(train_env._tasks)):
    # Cycle to task i
    for _ in range(i + 1):
        obs = train_env.reset()

    train_prompts.append({
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs.instruction},
        ],
        "task_idx": i,
        "task_id": obs.task_id,
        "difficulty": obs.metadata.get("difficulty", ""),
    })

# Repeat prompts to create enough training data (GRPO needs many samples)
# Weight harder tasks more
expanded = []
for p in train_prompts:
    repeat = {"simple": 5, "medium": 10, "complex": 15, "edge_case": 15}.get(p["difficulty"], 10)
    expanded.extend([p] * repeat)

dataset = Dataset.from_list(expanded)

print(f"Tasks: {len(train_prompts)}")
print(f"Training samples (with repeats): {len(expanded)}")
print(f"\nSample prompt:")
print(f"  Task: {train_prompts[14]['task_id']}")
print(f"  Difficulty: {train_prompts[14]['difficulty']}")
print(f"  Instruction: {train_prompts[14]['prompt'][1]['content'][:100]}...")

# Calculate max prompt length for GRPO config
maximum_length = max(
    len(tokenizer.apply_chat_template(p["prompt"], add_generation_prompt=True))
    for p in train_prompts
)
print(f"\nMax prompt token length: {maximum_length}")

Tasks: 77
Training samples (with repeats): 825

Sample prompt:
  Task: task_0043
  Difficulty: medium
  Instruction: Initiate offboarding for Henrik Becker (emp_0069) who being terminated for policy violations. Revoke...

Max prompt token length: 3605


Let's see what the **base model** (before RL training) generates:

In [7]:
# Test base model on a medium task
test_prompt = train_prompts[14]["prompt"]  # Medium onboarding task
print(f"Task: {train_prompts[14]['task_id']} — {test_prompt[1]['content']}\n")

text = tokenizer.apply_chat_template(
    test_prompt,
    tokenize=False,
    add_generation_prompt=True,
)

from transformers import TextStreamer

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    temperature=0.1,
    max_new_tokens=512,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

Unsloth: Input IDs of shape torch.Size([1, 3556]) with length 3556 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Task: task_0043 — Initiate offboarding for Henrik Becker (emp_0069) who being terminated for policy violations. Revoke their system access and notify IT.

,,
To on-board a new employee named John Doe, who will be joining the Engineering department as a Level 3 Software Engineer, please follow these steps:

1. Create a new employee record in the HR system.
2. Initiate an onboarding request for John Doe.
3. Assign an available laptop to John Doe.
4. Create IT accounts for John Doe (email, Slack, and VPN).
5. Complete the onboarding process.
{"tool": "hr_create_employee", "params": {"name": "John Doe", "department": "Engineering", "level": "L3", "role": "Software Engineer", "manager_id": "emp_0001", "location": "San Francisco", "date_of_joining": "2023-09-15"}}
{"tool": "onboarding_create_request", "params": {"employee_id": "hr_create_employee_response_emp_id"}}
{"tool": "it_get_available_assets", "params": {"asset_type": "laptop"}}
{"tool": "it_assign_asset", "params": {"asset_id": "it_g

## Designing Reward Functions

We need reward functions that evaluate the model's generated tool calls. Unlike the 2048 tutorial which used code sandboxing, here we:

1. **Parse** the model's output into JSON tool calls
2. **Replay** them against the HR environment
3. **Evaluate** using the task's rubric criteria

| Reward Function | Purpose | Score Range |
|-----------------|---------|-------------|
| `valid_json_reward` | Are the generated tool calls valid JSON? | -2.0 to +1.0 |
| `rubric_reward` | Does the sequence satisfy the task's rubric criteria? | -1.0 to +5.0 |
| `efficiency_reward` | Was the task completed without wasting steps? | -1.0 to +1.0 |

In [8]:
def extract_tool_calls(text):
    """Extract JSON tool calls from model output."""
    calls = []
    for match in re.finditer(r'\{[^{}]*\}', text):
        try:
            obj = json.loads(match.group())
            if "tool" in obj:
                calls.append(obj)
        except json.JSONDecodeError:
            continue
    return calls


def replay_tool_calls(task_idx, tool_calls):
    """Replay tool calls against a fresh environment and return evaluation."""
    replay_env = HROnboardingEnvironment(seed=42, max_steps=15)
    # Cycle to the right task
    for _ in range(task_idx + 1):
        replay_env.reset()

    steps = 0
    for tc in tool_calls:
        tool_name = tc.get("tool", "")
        params = tc.get("params", {})
        if tool_name == "__done__":
            break
        if steps >= 15:
            break
        action = HROnboardingAction(tool_name=tool_name, arguments=params)
        replay_env.step(action)
        steps += 1

    # Evaluate
    evaluator = RubricEvaluator()
    task = replay_env._current_task
    eval_result = evaluator.evaluate(task, replay_env.world.action_log)
    return eval_result, steps


# Test it
test_calls = [
    {"tool": "hr_create_employee", "params": {"name": "Priya Sharma", "department": "Engineering", "level": "L2", "role": "Software Engineer"}},
    {"tool": "onboarding_create_request", "params": {"employee_id": "emp_0201"}},
    {"tool": "__done__", "params": {}},
]
eval_result, steps = replay_tool_calls(14, test_calls)
print(f"Score: {eval_result['score']:.0%} ({eval_result['passed_count']}/{eval_result['total_criteria']})")
print(f"Passed: {eval_result['passed']}")
for c in eval_result["criteria_results"]:
    print(f"  [{'PASS' if c['passed'] else 'FAIL'}] {c['name']}: {c['description']}")

Score: 100% (7/7)
Passed: True
  [PASS] created_employee: Created employee record
  [PASS] correct_name: Used correct name
  [PASS] correct_dept: Assigned to correct department
  [PASS] correct_level: Set correct level
  [PASS] correct_role: Set correct role
  [PASS] initiated_onboarding: Created onboarding request
  [PASS] sequencing: Created employee before onboarding request


Now the actual reward functions that GRPO will call:

In [9]:
global PRINTER
PRINTER = 0


def valid_json_reward(completions, **kwargs):
    """Reward for generating valid JSON tool calls."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        calls = extract_tool_calls(response)
        if len(calls) == 0:
            scores.append(-2.0)  # No valid JSON at all
        elif any(c.get("tool") in [t["name"] for t in TOOL_DEFINITIONS] or c.get("tool") == "__done__" for c in calls):
            scores.append(1.0)   # Valid tool calls with known tool names
        else:
            scores.append(-0.5)  # JSON but unknown tool names
    return scores


def rubric_reward(completions, **kwargs):
    """Main reward: replay tool calls and evaluate against rubric."""
    global PRINTER
    prompt = kwargs.get("prompts", kwargs.get("prompt", []))
    scores = []
    for i, completion in enumerate(completions):
        response = completion[0]["content"]
        calls = extract_tool_calls(response)

        if len(calls) == 0:
            scores.append(-1.0)
            continue

        # Find the task_idx from the prompt
        # Match instruction text to find the task
        try:
            instruction = prompt[i][1]["content"] if len(prompt[i]) > 1 else ""
        except (IndexError, KeyError, TypeError):
            instruction = ""
        task_idx = 0
        for j, p in enumerate(train_prompts):
            if p["prompt"][1]["content"] == instruction:
                task_idx = p["task_idx"]
                break

        try:
            eval_result, steps = replay_tool_calls(task_idx, calls)
            score = eval_result["score"]

            # Scale: 0.0-1.0 rubric score → -1.0 to +5.0 reward
            reward = score * 6.0 - 1.0

            # Bonus for perfect score
            if eval_result["passed"]:
                reward += 2.0

            # Print every 10th completion for monitoring
            if PRINTER % 10 == 0:
                task = train_prompts[task_idx]
                print(f"\n--- [{task['task_id']}] [{task['difficulty']}] ---")
                print(f"Instruction: {instruction[:80]}...")
                print(f"Tool calls: {[c['tool'] for c in calls]}")
                print(f"Rubric: {eval_result['score']:.0%} ({eval_result['passed_count']}/{eval_result['total_criteria']})")
                print(f"Reward: {reward:.2f}")
            PRINTER += 1

            scores.append(reward)
        except Exception as e:
            print(f"Error replaying: {e}")
            scores.append(-1.0)

    return scores


def efficiency_reward(completions, **kwargs):
    """Reward for completing tasks efficiently (fewer steps = better)."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        calls = extract_tool_calls(response)
        actual_calls = [c for c in calls if c.get("tool") != "__done__"]
        n = len(actual_calls)

        if n == 0:
            scores.append(-1.0)
        elif n <= 3:
            scores.append(1.0)   # Very efficient
        elif n <= 6:
            scores.append(0.5)   # Reasonable
        elif n <= 10:
            scores.append(0.0)   # A bit wasteful
        else:
            scores.append(-0.5)  # Too many steps
    return scores

## Baseline Evaluation

Before training, let's evaluate the base model on a few tasks to establish a baseline:

In [10]:
def evaluate_model(model, tokenizer, task_indices=None, temperature=0.1):
    """Evaluate model on a set of tasks."""
    if task_indices is None:
        task_indices = list(range(len(train_prompts)))

    results = []
    for idx in task_indices:
        prompt_msgs = train_prompts[idx]["prompt"]
        text = tokenizer.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=temperature,
                do_sample=True,
            )
        response = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        calls = extract_tool_calls(response)
        if calls:
            eval_result, steps = replay_tool_calls(idx, calls)
            results.append({
                "task_id": train_prompts[idx]["task_id"],
                "difficulty": train_prompts[idx]["difficulty"],
                "score": eval_result["score"],
                "passed": eval_result["passed"],
                "steps": steps,
            })
        else:
            results.append({
                "task_id": train_prompts[idx]["task_id"],
                "difficulty": train_prompts[idx]["difficulty"],
                "score": 0.0,
                "passed": False,
                "steps": 0,
            })

    pass_count = sum(1 for r in results if r["passed"])
    mean_score = sum(r["score"] for r in results) / max(len(results), 1)

    print(f"\nResults: {pass_count}/{len(results)} passed ({pass_count/len(results):.1%})")
    print(f"Mean score: {mean_score:.3f}")

    # By difficulty
    for diff in ["simple", "medium", "complex", "edge_case"]:
        subset = [r for r in results if r["difficulty"] == diff]
        if subset:
            p = sum(1 for r in subset if r["passed"])
            s = sum(r["score"] for r in subset) / len(subset)
            print(f"  {diff:10s}: {p}/{len(subset)} pass, score={s:.2f}")

    return results


# Evaluate on a subset of tasks (to be fast)
eval_indices = [0, 1, 5, 10, 14, 15, 20, 24, 30, 40, 50, 55, 60, 65, 70]
print("=" * 50)
print("BASELINE EVALUATION (before training)")
print("=" * 50)
baseline_results = evaluate_model(model, tokenizer, eval_indices)

Unsloth: Input IDs of shape torch.Size([1, 3543]) with length 3543 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


BASELINE EVALUATION (before training)


Unsloth: Input IDs of shape torch.Size([1, 3544]) with length 3544 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 3550]) with length 3550 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 3546]) with length 3546 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 3540]) with length 3540 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 3585]) with length 3585 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([1, 35


Results: 0/15 passed (0.0%)
Mean score: 0.000
  simple    : 0/3 pass, score=0.00
  medium    : 0/6 pass, score=0.00
  complex   : 0/3 pass, score=0.00
  edge_case : 0/3 pass, score=0.00


## Training with GRPO

**Group Relative Policy Optimization (GRPO)** compares multiple generations for the same prompt and updates the policy to favor higher-reward outputs.

Key training parameters:
- `num_generations=2`: Generate 2 candidates per prompt to compute relative rewards
- `max_steps=300`: Training steps (increase for better results)
- `temperature=1.0`: Higher = more exploration during training

In [19]:
max_prompt_length

3606

In [20]:
max_prompt_length = maximum_length + 1
max_completion_length =  max_prompt_length

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # Increase to 4 for smoother training
    num_generations=2,              # Decrease if out of memory
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=300,                  # Increase to 600+ for better results
    save_steps=100,
    report_to="trackio",            # Can use "wandb" for Weights & Biases
    output_dir="outputs",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


In [21]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        valid_json_reward,
        rubric_reward,
        efficiency_reward,
    ],
    args=training_args,
    train_dataset=dataset,
)

### Start Training!

Training will take a while. Watch the reward column — it should gradually increase as the model learns to:
1. Generate valid JSON tool calls
2. Call the right tools for each task
3. Pass more rubric criteria

| Step | Training Loss | reward | reward_std | completion_length |
|------|--------------|--------|------------|-------------------|
| 1    | 0.000        | -1.5   | 0.5        | 400               |
| 50   | 0.001        | 0.5    | 1.2        | 350               |
| 150  | 0.002        | 2.0    | 1.5        | 300               |
| 300  | 0.001        | 3.5    | 1.0        | 250               |

In [22]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 825 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


* Trackio project initialized: huggingface
* Trackio metrics logged to: /home/jovyan/.cache/huggingface/trackio
* Created new run: dainty-sunset-0


/opt/conda/lib/python3.13/site-packages/websockets/legacy/__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
/opt/conda/lib/python3.13/site-packages/uvicorn/protocols/websockets/websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


Unsloth: Input IDs of shape torch.Size([2, 3579]) with length 3579 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Unsloth: Input IDs of shape torch.Size([2, 7185]) with length 7185 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


Unsloth: Will smartly offload gradients to save VRAM!


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function <built-in method gather of type object at 0x7f609a4e49e0>(*(FakeTensor(..., device='cuda:0', size=(((s47*s87 + 7)//8), 152064)),), **{'dim': -1, 'index': FakeTensor(..., device='cuda:0', size=(((s47*s61 + 7)//8), 1),
           dtype=torch.int64)}): got RuntimeError('Size does not match at dimension 0 expected index torch.Size([((s47*s61 + 7)//8), 1]) to be no larger than self torch.Size([((s47*s87 + 7)//8), 152064]) apart from dimension 1')

from user code:
   File "/home/jovyan/rl_hack/unsloth_compiled_cache/UnslothGRPOTrainer.py", line 137, in chunked_hidden_states_selective_log_softmax
    selected_logits = torch.gather(chunk_logits, dim=-1, index=chunk_index.unsqueeze(-1)).squeeze(-1)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


## Testing the Trained Model

Let's see what the RL-trained model generates compared to the base model:

In [ ]:
# Test on the same medium task
test_prompt = train_prompts[14]["prompt"]
print(f"Task: {train_prompts[14]['task_id']}")
print(f"Instruction: {test_prompt[1]['content']}\n")
print("Model output:")
print("-" * 40)

text = tokenizer.apply_chat_template(
    test_prompt,
    tokenize=False,
    add_generation_prompt=True,
)

from transformers import TextStreamer

inputs = tokenizer(text, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    temperature=0.1,
    max_new_tokens=512,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

# Evaluate the output
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
calls = extract_tool_calls(response)
print(f"\n\nTool calls extracted: {[c['tool'] for c in calls]}")

if calls:
    eval_result, steps = replay_tool_calls(14, calls)
    print(f"\nRubric score: {eval_result['score']:.0%} ({eval_result['passed_count']}/{eval_result['total_criteria']})")
    print(f"Passed: {eval_result['passed']}")
    for c in eval_result["criteria_results"]:
        print(f"  [{'PASS' if c['passed'] else 'FAIL'}] {c['name']}: {c['description']}")

## Post-Training Evaluation

Let's run the same evaluation as the baseline to measure improvement:

In [ ]:
print("=" * 50)
print("POST-TRAINING EVALUATION")
print("=" * 50)
trained_results = evaluate_model(model, tokenizer, eval_indices)

# Compare
baseline_pass = sum(1 for r in baseline_results if r["passed"])
trained_pass = sum(1 for r in trained_results if r["passed"])
baseline_score = sum(r["score"] for r in baseline_results) / len(baseline_results)
trained_score = sum(r["score"] for r in trained_results) / len(trained_results)

print(f"\n{'=' * 50}")
print(f"IMPROVEMENT SUMMARY")
print(f"{'=' * 50}")
print(f"Pass rate:  {baseline_pass}/{len(baseline_results)} → {trained_pass}/{len(trained_results)}  "
      f"({baseline_pass/len(baseline_results):.1%} → {trained_pass/len(trained_results):.1%})")
print(f"Mean score: {baseline_score:.3f} → {trained_score:.3f}  "
      f"({'+'if trained_score > baseline_score else ''}{trained_score - baseline_score:.3f})")

## Saving the Fine-tuned Model

Save the trained model for later use or push to Hugging Face Hub:

In [ ]:
# Save locally
model.save_pretrained_merged("outputs/hr_agent_final", tokenizer, save_method="merged_16bit")
print("Model saved to outputs/hr_agent_final")

# Push to Hugging Face Hub (uncomment and set your token)
# model.push_to_hub_merged(
#     "your-username/hr-onboarding-agent",
#     tokenizer,
#     save_method="merged_16bit",
#     token="hf_...",
# )

## Full Evaluation (All 77 Tasks)

For a comprehensive evaluation, run the model on all 77 tasks:

In [ ]:
print("=" * 50)
print("FULL EVALUATION (all 77 tasks)")
print("=" * 50)
full_results = evaluate_model(model, tokenizer)  # All tasks

# Save results
import os
os.makedirs("outputs", exist_ok=True)
with open("outputs/full_eval_trained.json", "w") as f:
    json.dump(full_results, f, indent=2)
print(f"\nResults saved to outputs/full_eval_trained.json")

## Conclusion

In this tutorial, we trained an LLM to automate HR workflows using reinforcement learning. Key concepts:

1. **OpenEnv** for standardized access to enterprise RL environments
2. **Rubric-based rewards** that verify tool usage, parameter correctness, and sequencing
3. **Multi-objective rewards** (valid JSON + rubric score + efficiency)
4. **GRPO** for policy optimization without a value network
5. **LoRA** for memory-efficient fine-tuning on consumer GPUs

### Key Results

| Metric | GPT-4o-mini (baseline) | Trained Model |
|--------|----------------------|---------------|
| Pass rate | 50.6% | TBD |
| Mean score | 0.791 | TBD |
| Simple tasks | 100% | TBD |
| Complex tasks | 12% | TBD |

### Further Improvements

- **More training steps** (600-1000) for better convergence
- **Larger LoRA rank** (16-32) for more model capacity
- **Multi-turn training** — feed tool results back and generate next calls
- **Curriculum learning** — start with simple tasks, gradually add complex ones

### Resources

- [HR Environment on HF Spaces](https://huggingface.co/spaces/devxpy/rl_hack)
- [OpenEnv Documentation](https://github.com/meta-pytorch/OpenEnv)
- [TRL GRPO Trainer](https://huggingface.co/docs/trl/main/en/grpo_trainer)
- [Unsloth RL Guide](https://docs.unsloth.ai/get-started/reinforcement-learning-rl-guide)

---

*This notebook uses [Unsloth](https://github.com/unslothai/unsloth) for memory-efficient training.*

**License:** Apache 2.0